# Extraction Batch Complète (VP + FP)

Cette étape réalise la détection classique de l'Objectif 2, génère des candidats, les compare au masque Ground Truth (VP=1, FP=0), extrait leurs caractéristiques radiomiques (Objectif 3) et sauvegarde la matrice finale.

In [ ]:
import sys
import os
from pathlib import Path

project_root = Path('.').resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.O2_classical_detection.detector import ClassicalDetector
from src.O2_classical_detection.connected_components import compute_overlap_with_gt

# === CONFIGURATION ===
processed_dir = project_root / 'data' / 'processed'
output_csv = processed_dir / 'radiomics_features_full.csv'

config = {
    'lung_hu_threshold': -500,
    'threshold_method': 'percentile',
    'threshold_percentile': 95,
    'min_hu': 30,
    'max_hu': 400,
    'min_candidate_volume': 20,
    'max_candidate_volume': 3000
}
detector = ClassicalDetector(config)

'''
Ex:
            'lung_hu_threshold': -400,
            'threshold_method': 'percentile',
            'threshold_percentile': 85,
            'min_hu': 20,
            'max_hu': 400,
            'min_candidate_volume': 20,
            'max_candidate_volume': 5000
TODO:
- Charger les volumes et masques GT
- Extraire les candidats VP et FP
- Extraire les features pour chaque candidat
- Sauvegarder dans un CSV
'''

print("✅ Configuration chargée.")

✅ Configuration chargée.


In [ ]:
# === BOUCLE SUR TOUS LES PATIENTS ===
patient_dirs = sorted([d for d in processed_dir.iterdir() if d.is_dir() and d.name.startswith('PAT')])
print(f'🚀 Lancement sur {len(patient_dirs)} patients...\n')

all_candidates = []

for i, p_dir in enumerate(patient_dirs):
    patient_id = p_dir.name
    vol_path = p_dir / 'volume.npy'
    mask_path = p_dir / 'mask.npy'
    
    if not vol_path.exists() or not mask_path.exists():
        print(f'⚠️ [{i+1}/{len(patient_dirs)}] {patient_id} : fichiers manquants, ignoré.')
        continue
    
    try:
        volume = np.load(vol_path)
        gt_mask = np.load(mask_path)
        
        # Étape 1 : Détection classique (seuillage + morpho + composantes connexes)
        result = detector.detect(volume)
        candidates_df = result['candidates_df']
        
        if len(candidates_df) == 0:
            print(f'[{i+1}/{len(patient_dirs)}] {patient_id} : aucun candidat détecté.')
            continue
        
        # Étape 2 : Labellisation VP/FP par IoU avec le Ground Truth
        candidates_df = compute_overlap_with_gt(candidates_df, gt_mask, iou_threshold=0.1)
        
        # Ajout de l'identifiant patient
        candidates_df['PatientID'] = patient_id
        
        n_vp = (candidates_df['is_positive'] == 1).sum()
        n_fp = (candidates_df['is_positive'] == 0).sum()
        print(f'[{i+1}/{len(patient_dirs)}] ✅ {patient_id}: {len(candidates_df)} candidats ({n_vp} VP, {n_fp} FP)')
        
        all_candidates.append(candidates_df)
        
    except Exception as e:
        print(f'❌ Erreur sur {patient_id}: {e}')

# === ASSEMBLAGE FINAL ===
if len(all_candidates) > 0:
    df_final = pd.concat(all_candidates, ignore_index=True)
    
    # Renommer le label pour cohérence avec le reste du projet
    df_final.rename(columns={'is_positive': 'has_embolism'}, inplace=True)
    
    # Réorganiser les colonnes
    id_cols = ['PatientID', 'region_id', 'has_embolism', 'iou_max']
    feature_cols = [c for c in df_final.columns if c not in id_cols]
    df_final = df_final[id_cols + feature_cols]
    
    df_final.to_csv(output_csv, index=False)
    
    print(f'\n{"="*60}')
    print(f'🎉 EXTRACTION TERMINÉE !')
    print(f'   Fichier : {output_csv}')
    print(f'   Échantillons totaux : {len(df_final)}')
    print(f'   VP (Vrais Positifs)  : {(df_final["has_embolism"]==1).sum()}')
    print(f'   FP (Faux Positifs)   : {(df_final["has_embolism"]==0).sum()}')
    print(f'{"="*60}')
else:
    print('🚫 Aucune donnée extraite.')

### Visualisation des résultats

In [ ]:
if output_csv.exists():
    df_final = pd.read_csv(output_csv)
    
    print('📊 Distribution des classes :')
    print(df_final['has_embolism'].value_counts().rename({0: 'FP (Faux Positif)', 1: 'VP (Vrai Positif)'}))
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 1. Distribution VP vs FP
    df_final['has_embolism'].value_counts().plot(kind='bar', ax=axes[0], color=['#e74c3c', '#2ecc71'])
    axes[0].set_title('Distribution VP / FP')
    axes[0].set_xticklabels(['Faux Positif (0)', 'Vrai Positif (1)'], rotation=0)

    # 2. Distribution du volume
    for label, color, name in [(1, '#2ecc71', 'VP'), (0, '#e74c3c', 'FP')]:
        subset = df_final[df_final['has_embolism'] == label]
        axes[1].hist(subset['volume_voxels'].dropna(), bins=30, alpha=0.6, color=color, label=name)
    axes[1].set_title('Volume (voxels)')
    axes[1].legend()

    # 3. Intensité moyenne
    for label, color, name in [(1, '#2ecc71', 'VP'), (0, '#e74c3c', 'FP')]:
        subset = df_final[df_final['has_embolism'] == label]
        axes[2].hist(subset['intensity_mean'].dropna(), bins=30, alpha=0.6, color=color, label=name)
    axes[2].set_title('Intensité Moyenne (HU)')
    axes[2].legend()

    plt.tight_layout()
    plt.show()